# Week 06 - AgriPulse Data Quality Checks

**ZENAIZ x BVRIT Hyderabad | Team 12 - AgriPulse**

### Objective
Evaluate the Week-05 Silver Candidate records using the approved AgriPulse DQ rule families, retain every applicable failure reason, route passing records to Trusted Silver, route failing records to Quarantine, and prove that no physical record is silently lost.

### Approved rule families
- `DQ-OBS-001` - Identity / duplicate / grain safety
- `DQ-REF-001` - Reference integrity
- `DQ-DAT-001` - Date scope
- `DQ-PRC-001` - Price validity
- `DQ-ARR-001` - Arrival validity
- `DQ-UNT-001` - Unit consistency
- `DQ-REL-001` - Price-arrival compatibility
- `DQ-EVT-001` - Streaming event quality

> **Important:** Do not invent private/configured price bounds, arrival bounds, approved unit lists, conversion factors, severity values, or event lateness policy. Enter only values approved by the team.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;

active_catalog,active_schema
workspace,default


## 1. Configuration

Change only the values that are approved for your AgriPulse project.

The table names below follow the Week-05 Candidate handoff. If your Week-05 notebook used different physical names, change them here before running the remaining cells.

In [0]:
# Approved AgriPulse Week-06 configuration
# Fill these ONLY with values approved by your team/rulebook.

MIN_REPORT_DATE = "2025-01-01"
MAX_REPORT_DATE = "2025-06-30"

# Private/configured bounds - DO NOT GUESS.
PRICE_MIN_ALLOWED = None
PRICE_MAX_ALLOWED = None
ARRIVAL_MIN_ALLOWED = None
ARRIVAL_MAX_ALLOWED = None

# Approved unit lists - DO NOT GUESS.
APPROVED_PRICE_UNITS = []
APPROVED_ARRIVAL_UNITS = []

# Candidate table names
MARKETS_CANDIDATE = "silver_markets_candidate"
COMMODITIES_CANDIDATE = "silver_commodities_candidate"
PRICES_CANDIDATE = "silver_prices_candidate"
ARRIVALS_CANDIDATE = "silver_arrivals_candidate"
DAILY_CANDIDATE = "silver_market_daily_candidate"

print("Configuration loaded.")
print("Date range:", MIN_REPORT_DATE, "to", MAX_REPORT_DATE)
print("Price bounds:", PRICE_MIN_ALLOWED, PRICE_MAX_ALLOWED)
print("Arrival bounds:", ARRIVAL_MIN_ALLOWED, ARRIVAL_MAX_ALLOWED)
print("Approved price units:", APPROVED_PRICE_UNITS)
print("Approved arrival units:", APPROVED_ARRIVAL_UNITS)

Configuration loaded.
Date range: 2025-01-01 to 2025-06-30
Price bounds: None None
Arrival bounds: None None
Approved price units: []
Approved arrival units: []


In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,agripulse_week03_bronze_demo_main,false
default,agripulse_week03_lineage_demo_view,false
default,bronze_arrivals,false
default,bronze_commodities,false
default,bronze_markets,false
default,bronze_prices,false
default,silver_arrivals_candidate,false
default,silver_commodities_candidate,false
default,silver_markets_candidate,false
default,silver_prices_candidate,false


## 2. Confirm the Week-05 Candidate handoff

The notebook must read the existing Candidate tables. It should not rebuild the Week-05 Silver transformations.

In [0]:
%sql
SELECT 'markets' AS entity, COUNT(*) AS candidate_rows
FROM silver_markets_candidate
UNION ALL
SELECT 'commodities', COUNT(*) FROM silver_commodities_candidate
UNION ALL
SELECT 'prices', COUNT(*) FROM silver_prices_candidate
UNION ALL
SELECT 'arrivals', COUNT(*) FROM silver_arrivals_candidate;

entity,candidate_rows
markets,606
commodities,164
prices,96125
arrivals,88220


In [0]:
%sql
DESCRIBE silver_markets_candidate;

col_name,data_type,comment
market_record_id,string,null
market_id,string,null
market_name,string,null
raw_market_name,string,null
market_type,string,null
district_id,string,null
district_name,string,null
state_code,string,null
state_name,string,null
latitude,double,null


In [0]:
%sql
DESCRIBE silver_commodities_candidate;

col_name,data_type,comment
commodity_variety_record_id,string,null
commodity_variety_id,string,null
commodity_id,string,null
commodity_name,string,null
raw_commodity_name,string,null
category,string,null
variety_id,string,null
variety_name,string,null
raw_variety_name,string,null
price_unit,string,null


In [0]:
%sql
DESCRIBE silver_prices_candidate;

col_name,data_type,comment
price_record_id,string,null
market_id,string,null
commodity_id,string,null
variety_id,string,null
report_date,date,null
minimum_price,double,null
modal_price,double,null
maximum_price,double,null
price_unit,string,null
source_system,string,null


In [0]:
%sql
DESCRIBE silver_arrivals_candidate;

col_name,data_type,comment
arrival_record_id,string,null
market_id,string,null
commodity_id,string,null
variety_id,string,null
report_date,date,null
arrival_quantity,double,null
arrival_unit,string,null
source_system,string,null
source_file_version,string,null
source_row_number,int,null


## 3. Required physical IDs and grain

Expected physical IDs are normally:
- markets: `market_id`
- commodities: `commodity_id` + `variety_id`
- prices: `price_record_id`
- arrivals: `arrival_record_id`

Expected price/arrival business grain:

`market_id + commodity_id + variety_id + report_date`

Keep `_bronze_record_hash`, `_source_file_name`, and `run_id` (or the equivalent lineage fields from Week 5) in the outputs.

In [0]:
%sql
SELECT
    market_id,
    source_file,
    source_row_number,
    source_system
FROM silver_markets_candidate
LIMIT 5;

market_id,source_file,source_row_number,source_system
MKT0001,markets.csv,1,AGRIPULSE_SYNTHETIC_MODEL_B
MKT0002,markets.csv,2,AGRIPULSE_SYNTHETIC_MODEL_B
MKT0003,markets.csv,3,AGRIPULSE_SYNTHETIC_MODEL_B
MKT0004,markets.csv,4,AGRIPULSE_SYNTHETIC_MODEL_B
MKT0005,markets.csv,5,AGRIPULSE_SYNTHETIC_MODEL_B


In [0]:
%sql
SELECT
    price_record_id,
    market_id,
    commodity_id,
    variety_id,
    report_date,
    source_file,
    source_row_number,
    source_system
FROM silver_prices_candidate
LIMIT 5;

price_record_id,market_id,commodity_id,variety_id,report_date,source_file,source_row_number,source_system
PR0000001,MKT0354,CMD012,VAR0046,2025-04-18,prices.csv,1,AGRIPULSE_SYNTHETIC_MODEL_B
PR0000002,MKT0268,CMD037,VAR0148,2025-05-14,prices.csv,2,AGRIPULSE_SYNTHETIC_MODEL_B
PR0000003,MKT0265,CMD012,VAR0048,2025-01-15,prices.csv,3,AGRIPULSE_SYNTHETIC_MODEL_B
PR0000004,MKT0061,CMD001,VAR0004,2025-03-03,prices.csv,4,AGRIPULSE_SYNTHETIC_MODEL_B
PR0000005,MKT0406,CMD014,VAR0055,2025-03-19,prices.csv,5,AGRIPULSE_SYNTHETIC_MODEL_B


## 4. DQ-OBS-001 - Markets

FAIL when the physical/business identity is missing or duplicated.

Every rule is evaluated independently. A row that fails several rules keeps all rule IDs in one `rule_ids` field.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW markets_checked AS

WITH duplicate_ids AS (
    SELECT
        market_id,
        COUNT(*) AS occurrences
    FROM silver_markets_candidate
    WHERE market_id IS NOT NULL
      AND trim(market_id) <> ''
    GROUP BY market_id
)

SELECT
    m.*,

    -- DQ-OBS-001: market_id check
    CASE
        WHEN m.market_id IS NULL
          OR trim(m.market_id) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_id_check,

    -- DQ-OBS-001: duplicate market_id check
    CASE
        WHEN d.occurrences > 1
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_duplicate_check,

    -- DQ-REF-001: market reference fields
    CASE
        WHEN m.market_name IS NULL
          OR trim(m.market_name) = ''
          OR m.state_name IS NULL
          OR trim(m.state_name) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS ref_check,

    -- Overall DQ status
    CASE
        WHEN m.market_id IS NULL
          OR trim(m.market_id) = ''
          OR d.occurrences > 1
          OR m.market_name IS NULL
          OR trim(m.market_name) = ''
          OR m.state_name IS NULL
          OR trim(m.state_name) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS dq_status,

    -- Rule IDs
    concat_ws(', ',
        CASE
            WHEN m.market_id IS NULL
              OR trim(m.market_id) = ''
            THEN 'DQ-OBS-001'
        END,

        CASE
            WHEN d.occurrences > 1
            THEN 'DQ-OBS-001'
        END,

        CASE
            WHEN m.market_name IS NULL
              OR trim(m.market_name) = ''
              OR m.state_name IS NULL
              OR trim(m.state_name) = ''
            THEN 'DQ-REF-001'
        END
    ) AS rule_ids,

    -- Failure reason
    concat_ws('; ',
        CASE
            WHEN m.market_id IS NULL
              OR trim(m.market_id) = ''
            THEN 'market_id is null or blank'
        END,

        CASE
            WHEN d.occurrences > 1
            THEN concat(
                'duplicate market_id occurs ',
                d.occurrences,
                ' times'
            )
        END,

        CASE
            WHEN m.market_name IS NULL
              OR trim(m.market_name) = ''
            THEN 'market_name is null or blank'
        END,

        CASE
            WHEN m.state_name IS NULL
              OR trim(m.state_name) = ''
            THEN 'state_name is null or blank'
        END
    ) AS reason

FROM silver_markets_candidate m

LEFT JOIN duplicate_ids d
    ON m.market_id = d.market_id;

In [0]:
%sql
SELECT
    market_id,
    market_name,
    state_name,
    dq_status,
    rule_ids,
    reason
FROM markets_checked
ORDER BY dq_status DESC;

market_id,market_name,state_name,dq_status,rule_ids,reason
MKT0572,KHORDHA MANDI 02,ODISHA,PASS,,
MKT0506,RAJKOT MANDI 06,GUJARAT,PASS,,
MKT0582,MAYURBHANJ MANDI 02,ODISHA,PASS,,
MKT0185,AHMEDNAGAR MANDI 05,MAHARASHTRA,PASS,,
MKT0219,NAGPUR MANDI 09,MAHARASHTRA,PASS,,
MKT0250,COIMBATORE MANDI 10,TAMIL NADU,PASS,,
MKT0281,SALEM MANDI 01,TAMIL NADU,PASS,,
MKT0290,SALEM MANDI 10,TAMIL NADU,PASS,,
MKT0170,MYSURU MANDI 10,KARNATAKA,PASS,,
MKT0186,AHMEDNAGAR MANDI 06,MAHARASHTRA,PASS,,


In [0]:
%sql
DESCRIBE silver_commodities_candidate;

col_name,data_type,comment
commodity_variety_record_id,string,null
commodity_variety_id,string,null
commodity_id,string,null
commodity_name,string,null
raw_commodity_name,string,null
category,string,null
variety_id,string,null
variety_name,string,null
raw_variety_name,string,null
price_unit,string,null


## 5. DQ-OBS-001 / DQ-REF-001 / DQ-UNT-001 - Commodities

The approved reference grain uses `commodity_id + variety_id`; do not use `variety_id` alone as the commodity reference key.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW commodities_checked AS

WITH duplicate_ids AS (
    SELECT
        commodity_id,
        variety_id,
        COUNT(*) AS occurrences
    FROM silver_commodities_candidate
    WHERE commodity_id IS NOT NULL
      AND trim(commodity_id) <> ''
      AND variety_id IS NOT NULL
      AND trim(variety_id) <> ''
    GROUP BY commodity_id, variety_id
)

SELECT
    c.*,

    -- DQ-OBS-001: Check commodity and variety IDs
    CASE
        WHEN c.commodity_id IS NULL
          OR trim(c.commodity_id) = ''
          OR c.variety_id IS NULL
          OR trim(c.variety_id) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_id_check,

    -- DQ-OBS-001: Check duplicate commodity + variety
    CASE
        WHEN d.occurrences > 1
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_duplicate_check,

    -- DQ-UNT-001: Check price and arrival units
    CASE
        WHEN c.price_unit IS NULL
          OR trim(c.price_unit) = ''
          OR c.arrival_unit IS NULL
          OR trim(c.arrival_unit) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS unit_check,

    -- DQ-UNT-001: Check conversion factor
    CASE
        WHEN c.arrival_to_tonne_factor IS NULL
          OR c.arrival_to_tonne_factor <= 0
        THEN 'FAIL'
        ELSE 'PASS'
    END AS conversion_check,

    -- Overall DQ status
    CASE
        WHEN c.commodity_id IS NULL
          OR trim(c.commodity_id) = ''
          OR c.variety_id IS NULL
          OR trim(c.variety_id) = ''
          OR d.occurrences > 1
          OR c.price_unit IS NULL
          OR trim(c.price_unit) = ''
          OR c.arrival_unit IS NULL
          OR trim(c.arrival_unit) = ''
          OR c.arrival_to_tonne_factor IS NULL
          OR c.arrival_to_tonne_factor <= 0
        THEN 'FAIL'
        ELSE 'PASS'
    END AS dq_status,

    -- Rule IDs
    concat_ws(', ',

        CASE
            WHEN c.commodity_id IS NULL
              OR trim(c.commodity_id) = ''
              OR c.variety_id IS NULL
              OR trim(c.variety_id) = ''
            THEN 'DQ-OBS-001'
        END,

        CASE
            WHEN d.occurrences > 1
            THEN 'DQ-OBS-001'
        END,

        CASE
            WHEN c.price_unit IS NULL
              OR trim(c.price_unit) = ''
              OR c.arrival_unit IS NULL
              OR trim(c.arrival_unit) = ''
            THEN 'DQ-UNT-001'
        END,

        CASE
            WHEN c.arrival_to_tonne_factor IS NULL
              OR c.arrival_to_tonne_factor <= 0
            THEN 'DQ-UNT-001'
        END

    ) AS rule_ids,

    -- Failure reasons
    concat_ws('; ',

        CASE
            WHEN c.commodity_id IS NULL
              OR trim(c.commodity_id) = ''
              OR c.variety_id IS NULL
              OR trim(c.variety_id) = ''
            THEN 'commodity_id or variety_id is null or blank'
        END,

        CASE
            WHEN d.occurrences > 1
            THEN concat(
                'duplicate commodity + variety occurs ',
                d.occurrences,
                ' times'
            )
        END,

        CASE
            WHEN c.price_unit IS NULL
              OR trim(c.price_unit) = ''
            THEN 'price_unit is null or blank'
        END,

        CASE
            WHEN c.arrival_unit IS NULL
              OR trim(c.arrival_unit) = ''
            THEN 'arrival_unit is null or blank'
        END,

        CASE
            WHEN c.arrival_to_tonne_factor IS NULL
              OR c.arrival_to_tonne_factor <= 0
            THEN 'arrival_to_tonne_factor is null or not positive'
        END

    ) AS reason

FROM silver_commodities_candidate c

LEFT JOIN duplicate_ids d
    ON c.commodity_id = d.commodity_id
   AND c.variety_id = d.variety_id;

In [0]:
%sql
SELECT
    commodity_variety_record_id,
    commodity_variety_id,
    commodity_id,
    commodity_name,
    variety_id,
    variety_name,
    price_unit,
    arrival_unit,
    arrival_to_tonne_factor,
    dq_status,
    rule_ids,
    reason
FROM commodities_checked
ORDER BY dq_status DESC;

commodity_variety_record_id,commodity_variety_id,commodity_id,commodity_name,variety_id,variety_name,price_unit,arrival_unit,arrival_to_tonne_factor,dq_status,rule_ids,reason
CVR00146,CV0146,CMD037,APPLE,VAR0146,GRADE A,INR_PER_QUINTAL,QUINTAL,0.1,PASS,,
CVR00123,CV0123,CMD031,RED CHILLI,VAR0123,LOCAL,INR_PER_QUINTAL,TONNE,1.0,PASS,,
CVR00158,CV0158,CMD040,GRAPES,VAR0158,GRADE A,INR_PER_QUINTAL,QUINTAL,0.1,PASS,,
CVR00074,CV0074,CMD019,MASOOR DAL,VAR0074,GRADE A,INR_PER_QUINTAL,QUINTAL,0.1,PASS,,
CVR00052,CV0052,CMD013,BAJRA,VAR0052,PREMIUM,INR_PER_QUINTAL,TONNE,1.0,PASS,,
CVR00060,CV0060,CMD015,CHANA DAL,VAR0060,PREMIUM,INR_PER_QUINTAL,TONNE,1.0,PASS,,
CVR00068,CV0068,CMD017,MOONG DAL,VAR0068,PREMIUM,INR_PER_QUINTAL,QUINTAL,0.1,PASS,,
CVR00077,CV0077,CMD020,GROUNDNUT,VAR0077,STANDARD,INR_PER_QUINTAL,QUINTAL,0.1,PASS,,
CVR00019,CV0019,CMD005,CABBAGE,VAR0019,LOCAL,INR_PER_QUINTAL,TONNE,1.0,PASS,,
CVR00045,CV0045,CMD012,JOWAR,VAR0045,STANDARD,INR_PER_QUINTAL,TONNE,1.0,PASS,,


## 6. DQ-OBS-001 / DQ-REF-001 / DQ-DAT-001 / DQ-PRC-001 / DQ-UNT-001 - Prices

Price checks:
- physical ID present;
- full business grain unique;
- reference keys present;
- `report_date` is within the approved date scope;
- minimum/modal/maximum are non-null where required, non-negative, and ordered;
- configured bounds are applied only when approved values are entered;
- configured approved unit list is applied only when populated.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW prices_checked AS
WITH duplicate_grain AS (
    SELECT
        market_id,
        commodity_id,
        variety_id,
        report_date,
        COUNT(*) AS occurrences
    FROM silver_prices_candidate
    GROUP BY market_id, commodity_id, variety_id, report_date
)
SELECT
    p.*,

    CASE
        WHEN p.price_record_id IS NULL OR trim(p.price_record_id) = ''
        THEN 'FAIL' ELSE 'PASS'
    END AS obs_id_check,

    CASE
        WHEN g.occurrences > 1 THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_grain_check,

    CASE
        WHEN p.market_id IS NULL OR trim(p.market_id) = ''
          OR p.commodity_id IS NULL OR trim(p.commodity_id) = ''
          OR p.variety_id IS NULL OR trim(p.variety_id) = ''
        THEN 'FAIL' ELSE 'PASS'
    END AS ref_check,

    CASE
        WHEN p.report_date IS NULL
          OR p.report_date < DATE('2025-01-01')
          OR p.report_date > DATE('2025-06-30')
        THEN 'FAIL' ELSE 'PASS'
    END AS date_check,

    CASE
        WHEN p.minimum_price IS NULL
          OR p.modal_price IS NULL
          OR p.maximum_price IS NULL
          OR p.minimum_price < 0
          OR p.modal_price < 0
          OR p.maximum_price < 0
          OR p.minimum_price > p.modal_price
          OR p.modal_price > p.maximum_price
        THEN 'FAIL' ELSE 'PASS'
    END AS price_order_check,

    CASE
        WHEN p.price_unit IS NULL OR trim(p.price_unit) = ''
        THEN 'FAIL' ELSE 'PASS'
    END AS unit_check,

    concat_ws(', ',
        CASE WHEN p.price_record_id IS NULL OR trim(p.price_record_id) = ''
             THEN 'DQ-OBS-001' END,
        CASE WHEN g.occurrences > 1 THEN 'DQ-OBS-001' END,
        CASE WHEN p.market_id IS NULL OR trim(p.market_id) = ''
                   OR p.commodity_id IS NULL OR trim(p.commodity_id) = ''
                   OR p.variety_id IS NULL OR trim(p.variety_id) = ''
             THEN 'DQ-REF-001' END,
        CASE WHEN p.report_date IS NULL
                   OR p.report_date < DATE('2025-01-01')
                   OR p.report_date > DATE('2025-06-30')
             THEN 'DQ-DAT-001' END,
        CASE WHEN p.minimum_price IS NULL
                   OR p.modal_price IS NULL
                   OR p.maximum_price IS NULL
                   OR p.minimum_price < 0
                   OR p.modal_price < 0
                   OR p.maximum_price < 0
                   OR p.minimum_price > p.modal_price
                   OR p.modal_price > p.maximum_price
             THEN 'DQ-PRC-001' END,
        CASE WHEN p.price_unit IS NULL OR trim(p.price_unit) = ''
             THEN 'DQ-UNT-001' END
    ) AS rule_ids,

    concat_ws('; ',
        CASE WHEN p.price_record_id IS NULL OR trim(p.price_record_id) = ''
             THEN 'price_record_id is null or blank' END,
        CASE WHEN g.occurrences > 1
             THEN concat('duplicate full price grain occurs ', g.occurrences, ' times') END,
        CASE WHEN p.market_id IS NULL OR trim(p.market_id) = ''
                   OR p.commodity_id IS NULL OR trim(p.commodity_id) = ''
                   OR p.variety_id IS NULL OR trim(p.variety_id) = ''
             THEN 'required market/commodity/variety reference key is missing' END,
        CASE WHEN p.report_date IS NULL
                   OR p.report_date < DATE('2025-01-01')
                   OR p.report_date > DATE('2025-06-30')
             THEN 'report_date is outside approved scope or cannot be parsed' END,
        CASE WHEN p.minimum_price IS NULL
                   OR p.modal_price IS NULL
                   OR p.maximum_price IS NULL
                   OR p.minimum_price < 0
                   OR p.modal_price < 0
                   OR p.maximum_price < 0
                   OR p.minimum_price > p.modal_price
                   OR p.modal_price > p.maximum_price
             THEN 'price values are null/negative or violate min <= modal <= max' END,
        CASE WHEN p.price_unit IS NULL OR trim(p.price_unit) = ''
             THEN 'price_unit is null or blank' END
    ) AS reason

FROM silver_prices_candidate p
LEFT JOIN duplicate_grain g
    ON p.market_id = g.market_id
   AND p.commodity_id = g.commodity_id
   AND p.variety_id = g.variety_id
   AND p.report_date = g.report_date;

In [0]:
%sql
SELECT
    price_record_id,
    market_id,
    commodity_id,
    variety_id,
    report_date,
    minimum_price,
    modal_price,
    maximum_price,
    price_unit,

    CASE
        WHEN rule_ids IS NULL OR trim(rule_ids) = ''
        THEN 'PASS'
        ELSE 'FAIL'
    END AS dq_status,

    rule_ids,
    reason

FROM prices_checked

WHERE rule_ids IS NOT NULL
  AND trim(rule_ids) <> ''

LIMIT 100;

price_record_id,market_id,commodity_id,variety_id,report_date,minimum_price,modal_price,maximum_price,price_unit,dq_status,rule_ids,reason
PR0000230,MKT0343,CMD024,VAR0096,2025-06-16,-12234.13,12531.43,12813.98,INR_PER_QUINTAL,FAIL,DQ-PRC-001,price values are null/negative or violate min <= modal <= max
PR0000255,MKT0296,CMD030,VAR0120,2025-01-22,23104.86,26587.4,26087.4,INR_PER_QUINTAL,FAIL,DQ-PRC-001,price values are null/negative or violate min <= modal <= max
PR0000670,MKT0237,CMD012,VAR0048,2025-03-31,-3635.84,3759.18,3927.25,INR_PER_QUINTAL,FAIL,DQ-PRC-001,price values are null/negative or violate min <= modal <= max
PR0000881,MKT0314,CMD013,VAR0051,2025-01-11,3374.46,3592.69,3756.66,INR_PER_QUINTAL,FAIL,DQ-OBS-001,duplicate full price grain occurs 2 times
PR0001285,MKT0414,CMD032,VAR0128,2025-03-08,-9735.98,10292.12,10753.31,INR_PER_QUINTAL,FAIL,DQ-PRC-001,price values are null/negative or violate min <= modal <= max
PR0001473,MKT0141,CMD019,VAR0073,2025-02-09,6940.61,7299.34,7807.91,INR_PER_QUINTAL,FAIL,DQ-OBS-001,duplicate full price grain occurs 2 times
PR0001523,MKT0258,CMD005,VAR0018,2025-06-05,-1321.43,1376.25,1448.1,INR_PER_QUINTAL,FAIL,DQ-PRC-001,price values are null/negative or violate min <= modal <= max
PR0001575,MKT0174,CMD009,VAR0034,2025-05-20,3219.58,3394.13,3599.02,INR_PER_QUINTAL,FAIL,DQ-OBS-001,duplicate full price grain occurs 2 times
PR0001616,MKT0344,CMD026,VAR0103,2026-12-31,522.29,540.68,555.68,INR_PER_QUINTAL,FAIL,DQ-DAT-001,report_date is outside approved scope or cannot be parsed
PR0001620,MKT0585,CMD037,VAR0145,2026-12-31,10862.26,11058.0,11330.42,INR_PER_QUINTAL,FAIL,DQ-DAT-001,report_date is outside approved scope or cannot be parsed


### Optional configured price-bound check

The SQL above implements the universally safe checks from the approved rule family. If your team has approved private minimum/maximum price bounds, add them to the configuration and apply them here rather than guessing them.

In [0]:
# Optional Python-side validation of configured values.
# This cell does not invent values.

if PRICE_MIN_ALLOWED is None or PRICE_MAX_ALLOWED is None:
    print("Price bounds are not configured. Do not invent them.")
else:
    print("Approved price bounds are configured:",
          PRICE_MIN_ALLOWED, "to", PRICE_MAX_ALLOWED)

Price bounds are not configured. Do not invent them.


## 7. DQ-OBS-001 / DQ-REF-001 / DQ-DAT-001 / DQ-ARR-001 / DQ-UNT-001 - Arrivals

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW arrivals_checked AS
WITH duplicate_grain AS (
    SELECT
        market_id,
        commodity_id,
        variety_id,
        report_date,
        COUNT(*) AS occurrences
    FROM silver_arrivals_candidate
    GROUP BY market_id, commodity_id, variety_id, report_date
)
SELECT
    a.*,

    CASE
        WHEN a.arrival_record_id IS NULL OR trim(a.arrival_record_id) = ''
        THEN 'FAIL' ELSE 'PASS'
    END AS obs_id_check,

    CASE
        WHEN g.occurrences > 1 THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_grain_check,

    CASE
        WHEN a.market_id IS NULL OR trim(a.market_id) = ''
          OR a.commodity_id IS NULL OR trim(a.commodity_id) = ''
          OR a.variety_id IS NULL OR trim(a.variety_id) = ''
        THEN 'FAIL' ELSE 'PASS'
    END AS ref_check,

    CASE
        WHEN a.report_date IS NULL
          OR a.report_date < DATE('2025-01-01')
          OR a.report_date > DATE('2025-06-30')
        THEN 'FAIL' ELSE 'PASS'
    END AS date_check,

    CASE
        WHEN a.arrival_quantity IS NULL
          OR a.arrival_quantity < 0
        THEN 'FAIL' ELSE 'PASS'
    END AS arrival_check,

    CASE
        WHEN a.arrival_unit IS NULL OR trim(a.arrival_unit) = ''
        THEN 'FAIL' ELSE 'PASS'
    END AS unit_check,

    concat_ws(', ',
        CASE WHEN a.arrival_record_id IS NULL OR trim(a.arrival_record_id) = ''
             THEN 'DQ-OBS-001' END,
        CASE WHEN g.occurrences > 1 THEN 'DQ-OBS-001' END,
        CASE WHEN a.market_id IS NULL OR trim(a.market_id) = ''
                   OR a.commodity_id IS NULL OR trim(a.commodity_id) = ''
                   OR a.variety_id IS NULL OR trim(a.variety_id) = ''
             THEN 'DQ-REF-001' END,
        CASE WHEN a.report_date IS NULL
                   OR a.report_date < DATE('2025-01-01')
                   OR a.report_date > DATE('2025-06-30')
             THEN 'DQ-DAT-001' END,
        CASE WHEN a.arrival_quantity IS NULL
                   OR a.arrival_quantity < 0
             THEN 'DQ-ARR-001' END,
        CASE WHEN a.arrival_unit IS NULL OR trim(a.arrival_unit) = ''
             THEN 'DQ-UNT-001' END
    ) AS rule_ids,

    concat_ws('; ',
        CASE WHEN a.arrival_record_id IS NULL OR trim(a.arrival_record_id) = ''
             THEN 'arrival_record_id is null or blank' END,
        CASE WHEN g.occurrences > 1
             THEN concat('duplicate full arrival grain occurs ', g.occurrences, ' times') END,
        CASE WHEN a.market_id IS NULL OR trim(a.market_id) = ''
                   OR a.commodity_id IS NULL OR trim(a.commodity_id) = ''
                   OR a.variety_id IS NULL OR trim(a.variety_id) = ''
             THEN 'required market/commodity/variety reference key is missing' END,
        CASE WHEN a.report_date IS NULL
                   OR a.report_date < DATE('2025-01-01')
                   OR a.report_date > DATE('2025-06-30')
             THEN 'report_date is outside approved scope or cannot be parsed' END,
        CASE WHEN a.arrival_quantity IS NULL OR a.arrival_quantity < 0
             THEN 'arrival_quantity is null or negative' END,
        CASE WHEN a.arrival_unit IS NULL OR trim(a.arrival_unit) = ''
             THEN 'arrival_unit is null or blank' END
    ) AS reason

FROM silver_arrivals_candidate a
LEFT JOIN duplicate_grain g
    ON a.market_id = g.market_id
   AND a.commodity_id = g.commodity_id
   AND a.variety_id = g.variety_id
   AND a.report_date = g.report_date;

In [0]:
%sql
SELECT
    arrival_record_id,
    market_id,
    commodity_id,
    variety_id,
    report_date,
    arrival_quantity,
    arrival_unit,

    CASE
        WHEN rule_ids IS NULL OR trim(rule_ids) = ''
        THEN 'PASS'
        ELSE 'FAIL'
    END AS dq_status,

    rule_ids,
    reason

FROM arrivals_checked

WHERE rule_ids IS NOT NULL
  AND trim(rule_ids) <> ''

LIMIT 100;

arrival_record_id,market_id,commodity_id,variety_id,report_date,arrival_quantity,arrival_unit,dq_status,rule_ids,reason
AR0000005,MKT0282,CMD009,VAR0033,2025-02-25,36.254,TONNE,FAIL,DQ-OBS-001,duplicate full arrival grain occurs 2 times
AR0000215,MKT0206,CMD028,VAR0109,2025-05-29,19.791,TONNE,FAIL,DQ-OBS-001,duplicate full arrival grain occurs 2 times
AR0000333,MKT0506,CMD009,VAR0034,2025-02-12,-25.339,TONNE,FAIL,DQ-ARR-001,arrival_quantity is null or negative
AR0000475,MKT0348,null,VAR0127,2025-06-03,30.481,TONNE,FAIL,DQ-REF-001,required market/commodity/variety reference key is missing
AR0000598,MKT0124,null,VAR0012,2025-04-25,73.596,TONNE,FAIL,DQ-REF-001,required market/commodity/variety reference key is missing
AR0000852,MKT0153,null,VAR0105,2025-06-11,33.835,TONNE,FAIL,DQ-REF-001,required market/commodity/variety reference key is missing
AR0001053,MKT0071,CMD007,VAR0025,2025-06-03,-110.838,TONNE,FAIL,DQ-ARR-001,arrival_quantity is null or negative
AR0001157,MKT0468,null,VAR0052,2025-05-30,65.072,TONNE,FAIL,DQ-REF-001,required market/commodity/variety reference key is missing
AR0001189,MKT0560,null,VAR0150,2025-03-16,16.442,TONNE,FAIL,DQ-REF-001,required market/commodity/variety reference key is missing
AR0001222,MKT0150,CMD009,VAR0035,2027-01-01,255.568,QUINTAL,FAIL,DQ-DAT-001,report_date is outside approved scope or cannot be parsed


## 8. Create Trusted Silver and Quarantine tables

Routing rule:

- `rule_ids` empty -> Trusted Silver
- one or more rule IDs -> matching Quarantine

No record is deleted, and a record failing several rules remains one physical quarantine record with all reasons retained.

In [0]:
%sql
CREATE OR REPLACE TABLE silver_markets_trusted AS
SELECT * FROM markets_checked
WHERE rule_ids = '';

CREATE OR REPLACE TABLE quarantine_markets AS
SELECT *,
       current_timestamp() AS quarantine_ts,
       'OPEN' AS rework_status
FROM markets_checked
WHERE rule_ids <> '';

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW commodities_checked AS

WITH duplicate_ids AS (
    SELECT
        commodity_id,
        variety_id,
        COUNT(*) AS occurrences
    FROM silver_commodities_candidate
    WHERE commodity_id IS NOT NULL
      AND trim(commodity_id) <> ''
      AND variety_id IS NOT NULL
      AND trim(variety_id) <> ''
    GROUP BY commodity_id, variety_id
)

SELECT
    c.*,

    CASE
        WHEN c.commodity_id IS NULL
          OR trim(c.commodity_id) = ''
          OR c.variety_id IS NULL
          OR trim(c.variety_id) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_id_check,

    CASE
        WHEN d.occurrences > 1
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_duplicate_check,

    CASE
        WHEN c.price_unit IS NULL
          OR trim(c.price_unit) = ''
          OR c.arrival_unit IS NULL
          OR trim(c.arrival_unit) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS unit_check,

    CASE
        WHEN c.arrival_to_tonne_factor IS NULL
          OR c.arrival_to_tonne_factor <= 0
        THEN 'FAIL'
        ELSE 'PASS'
    END AS conversion_check,

    CASE
        WHEN c.commodity_id IS NULL
          OR trim(c.commodity_id) = ''
          OR c.variety_id IS NULL
          OR trim(c.variety_id) = ''
          OR d.occurrences > 1
          OR c.price_unit IS NULL
          OR trim(c.price_unit) = ''
          OR c.arrival_unit IS NULL
          OR trim(c.arrival_unit) = ''
          OR c.arrival_to_tonne_factor IS NULL
          OR c.arrival_to_tonne_factor <= 0
        THEN 'FAIL'
        ELSE 'PASS'
    END AS dq_status,

    concat_ws(', ',
        CASE
            WHEN c.commodity_id IS NULL
              OR trim(c.commodity_id) = ''
              OR c.variety_id IS NULL
              OR trim(c.variety_id) = ''
            THEN 'DQ-OBS-001'
        END,

        CASE
            WHEN d.occurrences > 1
            THEN 'DQ-OBS-001'
        END,

        CASE
            WHEN c.price_unit IS NULL
              OR trim(c.price_unit) = ''
              OR c.arrival_unit IS NULL
              OR trim(c.arrival_unit) = ''
            THEN 'DQ-UNT-001'
        END,

        CASE
            WHEN c.arrival_to_tonne_factor IS NULL
              OR c.arrival_to_tonne_factor <= 0
            THEN 'DQ-UNT-001'
        END
    ) AS rule_ids,

    concat_ws('; ',
        CASE
            WHEN c.commodity_id IS NULL
              OR trim(c.commodity_id) = ''
              OR c.variety_id IS NULL
              OR trim(c.variety_id) = ''
            THEN 'commodity_id or variety_id is null or blank'
        END,

        CASE
            WHEN d.occurrences > 1
            THEN concat(
                'duplicate commodity + variety occurs ',
                d.occurrences,
                ' times'
            )
        END,

        CASE
            WHEN c.price_unit IS NULL
              OR trim(c.price_unit) = ''
            THEN 'price_unit is null or blank'
        END,

        CASE
            WHEN c.arrival_unit IS NULL
              OR trim(c.arrival_unit) = ''
            THEN 'arrival_unit is null or blank'
        END,

        CASE
            WHEN c.arrival_to_tonne_factor IS NULL
              OR c.arrival_to_tonne_factor <= 0
            THEN 'arrival_to_tonne_factor is null or not positive'
        END
    ) AS reason

FROM silver_commodities_candidate c

LEFT JOIN duplicate_ids d
    ON c.commodity_id = d.commodity_id
   AND c.variety_id = d.variety_id;

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM commodities_checked;

total_records
164


In [0]:
%sql

CREATE OR REPLACE TABLE silver_commodities_trusted AS
SELECT *
FROM commodities_checked
WHERE COALESCE(TRIM(rule_ids), '') = '';


CREATE OR REPLACE TABLE quarantine_commodities AS
SELECT
    *,
    current_timestamp() AS quarantine_ts,
    'OPEN' AS rework_status
FROM commodities_checked
WHERE COALESCE(TRIM(rule_ids), '') <> '';

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    'Candidate' AS table_name,
    COUNT(*) AS record_count
FROM silver_commodities_candidate

UNION ALL

SELECT
    'Trusted',
    COUNT(*)
FROM silver_commodities_trusted

UNION ALL

SELECT
    'Quarantine',
    COUNT(*)
FROM quarantine_commodities;

table_name,record_count
Candidate,164
Trusted,156
Quarantine,8


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW prices_checked AS

WITH duplicate_ids AS (
    SELECT
        price_record_id,
        COUNT(*) AS occurrences
    FROM silver_prices_candidate
    WHERE price_record_id IS NOT NULL
      AND TRIM(price_record_id) <> ''
    GROUP BY price_record_id
),

duplicate_grain AS (
    SELECT
        market_id,
        commodity_id,
        variety_id,
        report_date,
        COUNT(*) AS occurrences
    FROM silver_prices_candidate
    GROUP BY
        market_id,
        commodity_id,
        variety_id,
        report_date
)

SELECT
    p.*,

    -- DQ-OBS-001 : ID check
    CASE
        WHEN p.price_record_id IS NULL
          OR TRIM(p.price_record_id) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_id_check,

    -- DQ-OBS-001 : duplicate physical ID
    CASE
        WHEN d.occurrences > 1
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_duplicate_check,

    -- DQ-OBS-001 : duplicate business grain
    CASE
        WHEN g.occurrences > 1
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_grain_check,

    -- DQ-REF-001
    CASE
        WHEN p.market_id IS NULL OR TRIM(p.market_id) = ''
          OR p.commodity_id IS NULL OR TRIM(p.commodity_id) = ''
          OR p.variety_id IS NULL OR TRIM(p.variety_id) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS ref_check,

    -- DQ-DAT-001
    CASE
        WHEN p.report_date IS NULL
          OR TO_DATE(CAST(p.report_date AS STRING)) IS NULL
          OR TO_DATE(CAST(p.report_date AS STRING)) < DATE '2025-01-01'
          OR TO_DATE(CAST(p.report_date AS STRING)) > DATE '2025-06-30'
        THEN 'FAIL'
        ELSE 'PASS'
    END AS date_check,

    -- DQ-PRC-001
    CASE
        WHEN p.minimum_price IS NULL
          OR p.modal_price IS NULL
          OR p.maximum_price IS NULL
          OR p.minimum_price < 0
          OR p.modal_price < 0
          OR p.maximum_price < 0
          OR p.minimum_price > p.modal_price
          OR p.modal_price > p.maximum_price
        THEN 'FAIL'
        ELSE 'PASS'
    END AS price_order_check,

    -- DQ-UNT-001
    CASE
        WHEN p.price_unit IS NULL
          OR TRIM(p.price_unit) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS unit_check,

    -- Overall DQ status
    CASE
        WHEN p.price_record_id IS NULL
          OR TRIM(p.price_record_id) = ''
          OR d.occurrences > 1
          OR g.occurrences > 1
          OR p.market_id IS NULL
          OR TRIM(p.market_id) = ''
          OR p.commodity_id IS NULL
          OR TRIM(p.commodity_id) = ''
          OR p.variety_id IS NULL
          OR TRIM(p.variety_id) = ''
          OR p.report_date IS NULL
          OR TO_DATE(CAST(p.report_date AS STRING)) IS NULL
          OR TO_DATE(CAST(p.report_date AS STRING)) < DATE '2025-01-01'
          OR TO_DATE(CAST(p.report_date AS STRING)) > DATE '2025-06-30'
          OR p.minimum_price IS NULL
          OR p.modal_price IS NULL
          OR p.maximum_price IS NULL
          OR p.minimum_price < 0
          OR p.modal_price < 0
          OR p.maximum_price < 0
          OR p.minimum_price > p.modal_price
          OR p.modal_price > p.maximum_price
          OR p.price_unit IS NULL
          OR TRIM(p.price_unit) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS dq_status,

    -- Failed rule IDs
    CONCAT_WS(', ',
        CASE
            WHEN p.price_record_id IS NULL
              OR TRIM(p.price_record_id) = ''
              OR d.occurrences > 1
              OR g.occurrences > 1
            THEN 'DQ-OBS-001'
        END,

        CASE
            WHEN p.market_id IS NULL OR TRIM(p.market_id) = ''
              OR p.commodity_id IS NULL OR TRIM(p.commodity_id) = ''
              OR p.variety_id IS NULL OR TRIM(p.variety_id) = ''
            THEN 'DQ-REF-001'
        END,

        CASE
            WHEN p.report_date IS NULL
              OR TO_DATE(CAST(p.report_date AS STRING)) IS NULL
              OR TO_DATE(CAST(p.report_date AS STRING)) < DATE '2025-01-01'
              OR TO_DATE(CAST(p.report_date AS STRING)) > DATE '2025-06-30'
            THEN 'DQ-DAT-001'
        END,

        CASE
            WHEN p.minimum_price IS NULL
              OR p.modal_price IS NULL
              OR p.maximum_price IS NULL
              OR p.minimum_price < 0
              OR p.modal_price < 0
              OR p.maximum_price < 0
              OR p.minimum_price > p.modal_price
              OR p.modal_price > p.maximum_price
            THEN 'DQ-PRC-001'
        END,

        CASE
            WHEN p.price_unit IS NULL
              OR TRIM(p.price_unit) = ''
            THEN 'DQ-UNT-001'
        END
    ) AS rule_ids,

    -- Reason
    CONCAT_WS('; ',

        CASE
            WHEN p.price_record_id IS NULL
              OR TRIM(p.price_record_id) = ''
            THEN 'price_record_id is null or blank'
        END,

        CASE
            WHEN d.occurrences > 1
            THEN CONCAT(
                'duplicate price_record_id occurs ',
                d.occurrences,
                ' times'
            )
        END,

        CASE
            WHEN g.occurrences > 1
            THEN CONCAT(
                'duplicate price business grain occurs ',
                g.occurrences,
                ' times'
            )
        END,

        CASE
            WHEN p.market_id IS NULL OR TRIM(p.market_id) = ''
            THEN 'market_id is null or blank'
        END,

        CASE
            WHEN p.commodity_id IS NULL OR TRIM(p.commodity_id) = ''
            THEN 'commodity_id is null or blank'
        END,

        CASE
            WHEN p.variety_id IS NULL OR TRIM(p.variety_id) = ''
            THEN 'variety_id is null or blank'
        END,

        CASE
            WHEN p.report_date IS NULL
              OR TO_DATE(CAST(p.report_date AS STRING)) IS NULL
            THEN 'report_date cannot be parsed'
        END,

        CASE
            WHEN TO_DATE(CAST(p.report_date AS STRING)) < DATE '2025-01-01'
            THEN 'report_date before 2025-01-01'
        END,

        CASE
            WHEN TO_DATE(CAST(p.report_date AS STRING)) > DATE '2025-06-30'
            THEN 'report_date after 2025-06-30'
        END,

        CASE
            WHEN p.minimum_price IS NULL
              OR p.modal_price IS NULL
              OR p.maximum_price IS NULL
            THEN 'price value is null'
        END,

        CASE
            WHEN p.minimum_price < 0
              OR p.modal_price < 0
              OR p.maximum_price < 0
            THEN 'price value is negative'
        END,

        CASE
            WHEN p.minimum_price > p.modal_price
              OR p.modal_price > p.maximum_price
            THEN 'minimum_price <= modal_price <= maximum_price violated'
        END,

        CASE
            WHEN p.price_unit IS NULL OR TRIM(p.price_unit) = ''
            THEN 'price_unit is null or blank'
        END

    ) AS reason

FROM silver_prices_candidate p

LEFT JOIN duplicate_ids d
    ON p.price_record_id = d.price_record_id

LEFT JOIN duplicate_grain g
    ON p.market_id = g.market_id
   AND p.commodity_id = g.commodity_id
   AND p.variety_id = g.variety_id
   AND p.report_date = g.report_date;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_prices_trusted AS
SELECT *
FROM prices_checked
WHERE COALESCE(TRIM(rule_ids), '') = '';

CREATE OR REPLACE TABLE quarantine_prices AS
SELECT
    *,
    current_timestamp() AS quarantine_ts,
    'OPEN' AS rework_status
FROM prices_checked
WHERE COALESCE(TRIM(rule_ids), '') <> '';

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW arrivals_checked AS

WITH duplicate_ids AS (
    SELECT
        arrival_record_id,
        COUNT(*) AS occurrences
    FROM silver_arrivals_candidate
    WHERE arrival_record_id IS NOT NULL
      AND TRIM(arrival_record_id) <> ''
    GROUP BY arrival_record_id
),

duplicate_grain AS (
    SELECT
        market_id,
        commodity_id,
        variety_id,
        report_date,
        COUNT(*) AS occurrences
    FROM silver_arrivals_candidate
    GROUP BY
        market_id,
        commodity_id,
        variety_id,
        report_date
)

SELECT
    a.*,

    -- DQ-OBS-001 : Physical ID
    CASE
        WHEN a.arrival_record_id IS NULL
          OR TRIM(a.arrival_record_id) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_id_check,

    -- DQ-OBS-001 : Duplicate physical ID
    CASE
        WHEN d.occurrences > 1
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_duplicate_check,

    -- DQ-OBS-001 : Duplicate business grain
    CASE
        WHEN g.occurrences > 1
        THEN 'FAIL'
        ELSE 'PASS'
    END AS obs_grain_check,

    -- DQ-REF-001
    CASE
        WHEN a.market_id IS NULL OR TRIM(a.market_id) = ''
          OR a.commodity_id IS NULL OR TRIM(a.commodity_id) = ''
          OR a.variety_id IS NULL OR TRIM(a.variety_id) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS ref_check,

    -- DQ-DAT-001
    CASE
        WHEN a.report_date IS NULL
          OR TO_DATE(CAST(a.report_date AS STRING)) IS NULL
          OR TO_DATE(CAST(a.report_date AS STRING)) < DATE '2025-01-01'
          OR TO_DATE(CAST(a.report_date AS STRING)) > DATE '2025-06-30'
        THEN 'FAIL'
        ELSE 'PASS'
    END AS date_check,

    -- DQ-ARR-001
    CASE
        WHEN a.arrival_quantity IS NULL
          OR a.arrival_quantity < 0
        THEN 'FAIL'
        ELSE 'PASS'
    END AS arrival_check,

    -- DQ-UNT-001
    CASE
        WHEN a.arrival_unit IS NULL
          OR TRIM(a.arrival_unit) = ''
        THEN 'FAIL'
        ELSE 'PASS'
    END AS unit_check,

    -- Overall DQ status
    CASE
        WHEN a.arrival_record_id IS NULL
          OR TRIM(a.arrival_record_id) = ''
          OR d.occurrences > 1
          OR g.occurrences > 1

          OR a.market_id IS NULL
          OR TRIM(a.market_id) = ''

          OR a.commodity_id IS NULL
          OR TRIM(a.commodity_id) = ''

          OR a.variety_id IS NULL
          OR TRIM(a.variety_id) = ''

          OR a.report_date IS NULL
          OR TO_DATE(CAST(a.report_date AS STRING)) IS NULL
          OR TO_DATE(CAST(a.report_date AS STRING)) < DATE '2025-01-01'
          OR TO_DATE(CAST(a.report_date AS STRING)) > DATE '2025-06-30'

          OR a.arrival_quantity IS NULL
          OR a.arrival_quantity < 0

          OR a.arrival_unit IS NULL
          OR TRIM(a.arrival_unit) = ''

        THEN 'FAIL'
        ELSE 'PASS'
    END AS dq_status,

    -- Rule IDs
    CONCAT_WS(', ',

        CASE
            WHEN a.arrival_record_id IS NULL
              OR TRIM(a.arrival_record_id) = ''
              OR d.occurrences > 1
              OR g.occurrences > 1
            THEN 'DQ-OBS-001'
        END,

        CASE
            WHEN a.market_id IS NULL OR TRIM(a.market_id) = ''
              OR a.commodity_id IS NULL OR TRIM(a.commodity_id) = ''
              OR a.variety_id IS NULL OR TRIM(a.variety_id) = ''
            THEN 'DQ-REF-001'
        END,

        CASE
            WHEN a.report_date IS NULL
              OR TO_DATE(CAST(a.report_date AS STRING)) IS NULL
              OR TO_DATE(CAST(a.report_date AS STRING)) < DATE '2025-01-01'
              OR TO_DATE(CAST(a.report_date AS STRING)) > DATE '2025-06-30'
            THEN 'DQ-DAT-001'
        END,

        CASE
            WHEN a.arrival_quantity IS NULL
              OR a.arrival_quantity < 0
            THEN 'DQ-ARR-001'
        END,

        CASE
            WHEN a.arrival_unit IS NULL
              OR TRIM(a.arrival_unit) = ''
            THEN 'DQ-UNT-001'
        END

    ) AS rule_ids,

    -- Reason
    CONCAT_WS('; ',

        CASE
            WHEN a.arrival_record_id IS NULL
              OR TRIM(a.arrival_record_id) = ''
            THEN 'arrival_record_id is null or blank'
        END,

        CASE
            WHEN d.occurrences > 1
            THEN CONCAT(
                'duplicate arrival_record_id occurs ',
                d.occurrences,
                ' times'
            )
        END,

        CASE
            WHEN g.occurrences > 1
            THEN CONCAT(
                'duplicate arrival business grain occurs ',
                g.occurrences,
                ' times'
            )
        END,

        CASE
            WHEN a.market_id IS NULL OR TRIM(a.market_id) = ''
            THEN 'market_id is null or blank'
        END,

        CASE
            WHEN a.commodity_id IS NULL OR TRIM(a.commodity_id) = ''
            THEN 'commodity_id is null or blank'
        END,

        CASE
            WHEN a.variety_id IS NULL OR TRIM(a.variety_id) = ''
            THEN 'variety_id is null or blank'
        END,

        CASE
            WHEN a.report_date IS NULL
              OR TO_DATE(CAST(a.report_date AS STRING)) IS NULL
            THEN 'report_date cannot be parsed'
        END,

        CASE
            WHEN TO_DATE(CAST(a.report_date AS STRING)) < DATE '2025-01-01'
            THEN 'report_date before 2025-01-01'
        END,

        CASE
            WHEN TO_DATE(CAST(a.report_date AS STRING)) > DATE '2025-06-30'
            THEN 'report_date after 2025-06-30'
        END,

        CASE
            WHEN a.arrival_quantity IS NULL
            THEN 'arrival_quantity is null'
        END,

        CASE
            WHEN a.arrival_quantity < 0
            THEN 'arrival_quantity is negative'
        END,

        CASE
            WHEN a.arrival_unit IS NULL OR TRIM(a.arrival_unit) = ''
            THEN 'arrival_unit is null or blank'
        END

    ) AS reason

FROM silver_arrivals_candidate a

LEFT JOIN duplicate_ids d
    ON a.arrival_record_id = d.arrival_record_id

LEFT JOIN duplicate_grain g
    ON a.market_id = g.market_id
   AND a.commodity_id = g.commodity_id
   AND a.variety_id = g.variety_id
   AND a.report_date = g.report_date;

In [0]:
%sql
SELECT *
FROM arrivals_checked
LIMIT 10;

arrival_record_id,market_id,commodity_id,variety_id,report_date,arrival_quantity,arrival_unit,source_system,source_file_version,source_row_number,source_record_updated_at,load_batch_id,raw_market_name,raw_commodity_name,raw_variety_name,source_file,ingestion_time,obs_id_check,obs_duplicate_check,obs_grain_check,ref_check,date_check,arrival_check,unit_check,dq_status,rule_ids,reason
AR0000001,MKT0489,CMD003,VAR0012,2025-04-15,184.72,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,1,2026-07-01T04:00:00.000Z,BATCH_20260718_01,AHMEDABAD MANDI 09,POTATO,PREMIUM,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000002,MKT0139,CMD020,VAR0080,2025-02-11,472.294,QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,2,2026-07-02T05:00:00.000Z,BATCH_20260718_01,BALLARI MANDI 09,GROUNDNUT,PREMIUM,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000003,MKT0325,CMD040,VAR0157,2025-06-29,403.579,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,3,2026-07-03T06:00:00.000Z,BATCH_20260718_01,INDORE MKT. 05,GRAPES,STANDARD,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000004,MKT0005,CMD011,VAR0043,2025-04-21,60.794,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,4,2026-07-04T07:00:00.000Z,BATCH_20260718_01,ADILABAD MKT. 05,MAIZE,LOCAL,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000005,MKT0282,CMD009,VAR0033,2025-02-25,36.254,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,5,2026-07-05T08:00:00.000Z,BATCH_20260718_01,SALEM MANDI 02,RICE,STANDARD,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,FAIL,PASS,PASS,PASS,PASS,FAIL,DQ-OBS-001,duplicate arrival business grain occurs 2 times
AR0000006,MKT0353,CMD038,VAR0150,2025-02-26,50.368,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,6,2026-07-06T09:00:00.000Z,BATCH_20260718_01,UJJAIN MANDI 03,GUAVA,GRADE A,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000007,MKT0592,CMD007,VAR0026,2025-03-26,2500.562,QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,7,2026-07-07T10:00:00.000Z,BATCH_20260718_01,SAMBALPUR MANDI 02,OKRA,GRADE A,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000008,MKT0257,CMD039,VAR0154,2025-03-11,18.995,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,8,2026-07-08T11:00:00.000Z,BATCH_20260718_01,DINDIGUL MANDI 07,POMEGRANATE,GRADE A,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000009,MKT0174,CMD018,VAR0069,2025-05-31,27.222,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,9,2026-07-09T12:00:00.000Z,BATCH_20260718_01,TUMAKURU MANDI 04,URAD DAL,STANDARD,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000010,MKT0317,CMD033,VAR0130,2025-04-20,95.596,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,10,2026-07-10T04:00:00.000Z,BATCH_20260718_01,DEWAS MANDI 07,BANANA,GRADE A,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,


In [0]:
%sql
CREATE OR REPLACE TABLE silver_arrivals_trusted AS
SELECT *
FROM arrivals_checked
WHERE COALESCE(TRIM(rule_ids), '') = '';

CREATE OR REPLACE TABLE quarantine_arrivals AS
SELECT
    *,
    current_timestamp() AS quarantine_ts,
    'OPEN' AS rework_status
FROM arrivals_checked
WHERE COALESCE(TRIM(rule_ids), '') <> '';

num_affected_rows,num_inserted_rows


## 9. DQ results summary

The summary counts failing records, not rule-result rows. One physical record can fail multiple rules.

In [0]:
%sql
SELECT *
FROM markets_checked
LIMIT 10;

market_record_id,market_id,market_name,raw_market_name,market_type,district_id,district_name,state_code,state_name,latitude,longitude,active_flag,valid_from,source_system,source_record_version,source_row_number,source_file,ingestion_time,obs_id_check,obs_duplicate_check,ref_check,dq_status,rule_ids,reason
MR00001,MKT0001,ADILABAD MANDI 01,ADILABAD MANDI 01,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.07,72.08,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,1,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00002,MKT0002,ADILABAD MANDI 02,ADILABAD MANDI 02,APMC MAIN YARD,TS-D01,ADILABAD,TS,TELANGANA,15.14,72.16,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,2,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00003,MKT0003,ADILABAD MANDI 03,ADILABAD MANDI 03,REGULATED MARKET,TS-D01,ADILABAD,TS,TELANGANA,15.21,72.24,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,3,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00004,MKT0004,ADILABAD MANDI 04,ADILABAD MANDI 04,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.28,72.32,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,4,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00005,MKT0005,ADILABAD MANDI 05,ADILABAD MKT. 05,APMC MAIN YARD,TS-D01,ADILABAD,TS,TELANGANA,15.35,72.4,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,5,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00006,MKT0006,ADILABAD MANDI 06,ADILABAD MANDI 06,REGULATED MARKET,TS-D01,ADILABAD,TS,TELANGANA,15.42,72.48,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,6,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00007,MKT0007,ADILABAD MANDI 07,ADILABAD MANDI 07,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.49,72.56,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,7,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00008,MKT0008,ADILABAD MANDI 08,ADILABAD MANDI 08,APMC MAIN YARD,TS-D01,ADILABAD,TS,TELANGANA,15.56,72.64,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,8,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00009,MKT0009,ADILABAD MANDI 09,ADILABAD MANDI 09,REGULATED MARKET,TS-D01,ADILABAD,TS,TELANGANA,15.63,72.72,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,9,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00010,MKT0010,ADILABAD MANDI 10,ADILABAD MKT. 10,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.7,72.8,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,10,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,


In [0]:
%sql
SELECT COUNT(*) FROM commodities_checked;

COUNT(*)
164


In [0]:
%sql
SELECT COUNT(*) FROM prices_checked;

COUNT(*)
96125


In [0]:
%sql
SELECT COUNT(*) FROM arrivals_checked;

COUNT(*)
88220


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dq_results_summary AS

-- =========================
-- MARKETS
-- =========================

SELECT
    'markets' AS entity,
    'DQ-OBS-001' AS rule_id,
    COUNT(*) AS total_records,
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-OBS-001') > 0
            THEN 1 ELSE 0
        END
    ) AS failed_records
FROM markets_checked

UNION ALL

SELECT
    'markets',
    'DQ-REF-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-REF-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM markets_checked


-- =========================
-- COMMODITIES
-- =========================

UNION ALL

SELECT
    'commodities',
    'DQ-OBS-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-OBS-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM commodities_checked

UNION ALL

SELECT
    'commodities',
    'DQ-REF-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-REF-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM commodities_checked

UNION ALL

SELECT
    'commodities',
    'DQ-UNT-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-UNT-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM commodities_checked


-- =========================
-- PRICES
-- =========================

UNION ALL

SELECT
    'prices',
    'DQ-OBS-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-OBS-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM prices_checked

UNION ALL

SELECT
    'prices',
    'DQ-REF-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-REF-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM prices_checked

UNION ALL

SELECT
    'prices',
    'DQ-DAT-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-DAT-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM prices_checked

UNION ALL

SELECT
    'prices',
    'DQ-PRC-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-PRC-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM prices_checked

UNION ALL

SELECT
    'prices',
    'DQ-UNT-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-UNT-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM prices_checked


-- =========================
-- ARRIVALS
-- =========================

UNION ALL

SELECT
    'arrivals',
    'DQ-OBS-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-OBS-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM arrivals_checked

UNION ALL

SELECT
    'arrivals',
    'DQ-REF-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-REF-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM arrivals_checked

UNION ALL

SELECT
    'arrivals',
    'DQ-DAT-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-DAT-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM arrivals_checked

UNION ALL

SELECT
    'arrivals',
    'DQ-ARR-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-ARR-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM arrivals_checked

UNION ALL

SELECT
    'arrivals',
    'DQ-UNT-001',
    COUNT(*),
    SUM(
        CASE
            WHEN INSTR(rule_ids, 'DQ-UNT-001') > 0
            THEN 1 ELSE 0
        END
    )
FROM arrivals_checked;

In [0]:
%sql
SELECT
    entity,
    rule_id,
    total_records,
    COALESCE(failed_records, 0) AS failed_records,
    CASE
        WHEN COALESCE(failed_records, 0) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS status
FROM dq_results_summary
ORDER BY entity, rule_id;

entity,rule_id,total_records,failed_records,status
arrivals,DQ-ARR-001,88220,300,FAIL
arrivals,DQ-DAT-001,88220,220,FAIL
arrivals,DQ-OBS-001,88220,444,FAIL
arrivals,DQ-REF-001,88220,120,FAIL
arrivals,DQ-UNT-001,88220,0,PASS
commodities,DQ-OBS-001,164,8,FAIL
commodities,DQ-REF-001,164,0,PASS
commodities,DQ-UNT-001,164,0,PASS
markets,DQ-OBS-001,606,12,FAIL
markets,DQ-REF-001,606,0,PASS


## 10. Failed-record evidence

Run the following cells and capture screenshots. Use the actual output; do not fabricate examples.

In [0]:
%sql
SELECT
    price_record_id,
    market_id,
    commodity_id,
    variety_id,
    report_date,
    minimum_price,
    modal_price,
    maximum_price,
    price_unit,
    rule_ids,
    reason,
    source_file,
    source_file_version,
    source_row_number,
    source_system,
    quarantine_ts,
    rework_status
FROM quarantine_prices
ORDER BY quarantine_ts DESC
LIMIT 20;

price_record_id,market_id,commodity_id,variety_id,report_date,minimum_price,modal_price,maximum_price,price_unit,rule_ids,reason,source_file,source_file_version,source_row_number,source_system,quarantine_ts,rework_status
PR0002408,MKT0524,CMD011,VAR0043,2026-12-31,2185.54,2273.16,2360.51,INR_PER_QUINTAL,DQ-DAT-001,report_date after 2025-06-30,prices.csv,v1.0,2408,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN
PR0001836,MKT0038,CMD016,VAR0062,2025-02-07,-9614.54,9970.81,10508.03,INR_PER_QUINTAL,DQ-PRC-001,price value is negative,prices.csv,v1.0,1836,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN
PR0002114,MKT0509,CMD033,VAR0131,2025-02-14,2929.27,3069.29,3204.68,INR_PER_QUINTAL,DQ-OBS-001,duplicate price business grain occurs 2 times,prices.csv,v1.0,2114,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN
PR0002318,MKT0127,CMD034,VAR0134,2025-02-13,-5690.03,6023.73,6343.17,INR_PER_QUINTAL,DQ-PRC-001,price value is negative,prices.csv,v1.0,2318,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN
PR0001635,MKT0263,CMD009,VAR0033,2025-03-14,3207.85,4024.04,3524.04,INR_PER_QUINTAL,DQ-PRC-001,minimum_price <= modal_price <= maximum_price violated,prices.csv,v1.0,1635,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN
PR0001817,MKT0478,CMD009,VAR0036,2025-01-07,-3493.2,3654.76,3847.94,INR_PER_QUINTAL,DQ-PRC-001,price value is negative,prices.csv,v1.0,1817,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN
PR0001620,MKT0585,CMD037,VAR0145,2026-12-31,10862.26,11058.0,11330.42,INR_PER_QUINTAL,DQ-DAT-001,report_date after 2025-06-30,prices.csv,v1.0,1620,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN
PR0001944,MKT0140,CMD037,VAR0146,2025-05-01,9423.01,10503.59,10003.59,INR_PER_QUINTAL,DQ-PRC-001,minimum_price <= modal_price <= maximum_price violated,prices.csv,v1.0,1944,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN
PR0001616,MKT0344,CMD026,VAR0103,2026-12-31,522.29,540.68,555.68,INR_PER_QUINTAL,DQ-DAT-001,report_date after 2025-06-30,prices.csv,v1.0,1616,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN
PR0001575,MKT0174,CMD009,VAR0034,2025-05-20,3219.58,3394.13,3599.02,INR_PER_QUINTAL,DQ-OBS-001,duplicate price business grain occurs 2 times,prices.csv,v1.0,1575,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN


In [0]:
%sql

SELECT
    arrival_record_id,
    market_id,
    commodity_id,
    variety_id,
    report_date,
    arrival_quantity,
    arrival_unit,
    rule_ids,
    reason,
    source_file,
    source_file_version,
    source_row_number,
    source_system,
    quarantine_ts,
    rework_status
FROM quarantine_arrivals
ORDER BY quarantine_ts DESC
LIMIT 20;

arrival_record_id,market_id,commodity_id,variety_id,report_date,arrival_quantity,arrival_unit,rule_ids,reason,source_file,source_file_version,source_row_number,source_system,quarantine_ts,rework_status
AR0001982,MKT0072,CMD028,VAR0109,2025-01-25,63.109,TONNE,DQ-OBS-001,duplicate arrival business grain occurs 2 times,arrivals.csv,v1.0,1982,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T09:01:57.741Z,OPEN
AR0001344,MKT0499,CMD022,VAR0085,2025-02-13,-12.37,TONNE,DQ-ARR-001,arrival_quantity is negative,arrivals.csv,v1.0,1344,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T09:01:57.741Z,OPEN
AR0001445,MKT0202,CMD040,VAR0158,2025-05-31,-1261.545,QUINTAL,DQ-ARR-001,arrival_quantity is negative,arrivals.csv,v1.0,1445,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T09:01:57.741Z,OPEN
AR0001537,MKT0286,CMD022,VAR0085,null,17.278,TONNE,DQ-DAT-001,report_date cannot be parsed,arrivals.csv,v1.0,1537,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T09:01:57.741Z,OPEN
AR0001232,MKT0288,CMD006,VAR0022,2025-01-04,-119.687,TONNE,DQ-ARR-001,arrival_quantity is negative,arrivals.csv,v1.0,1232,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T09:01:57.741Z,OPEN
AR0001316,MKT0003,null,VAR0049,2025-02-09,22.293,TONNE,DQ-REF-001,commodity_id is null or blank,arrivals.csv,v1.0,1316,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T09:01:57.741Z,OPEN
AR0001222,MKT0150,CMD009,VAR0035,2027-01-01,255.568,QUINTAL,DQ-DAT-001,report_date after 2025-06-30,arrivals.csv,v1.0,1222,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T09:01:57.741Z,OPEN
AR0001437,MKT0115,CMD010,VAR0040,2025-04-04,-72.601,TONNE,DQ-ARR-001,arrival_quantity is negative,arrivals.csv,v1.0,1437,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T09:01:57.741Z,OPEN
AR0001189,MKT0560,null,VAR0150,2025-03-16,16.442,TONNE,DQ-REF-001,commodity_id is null or blank,arrivals.csv,v1.0,1189,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T09:01:57.741Z,OPEN
AR0001157,MKT0468,null,VAR0052,2025-05-30,65.072,TONNE,DQ-REF-001,commodity_id is null or blank,arrivals.csv,v1.0,1157,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T09:01:57.741Z,OPEN


### Evidence filenames

Save screenshots from Databricks as:
- `week06_dq_results.png` - DQ scorecard
- `week06_failed_records_sample.png` - actual quarantined records with rule IDs/reasons

## 11. Candidate = Trusted + Quarantine reconciliation

This is a mandatory control. The expected variance is **0** for every batch entity.

In [0]:
%sql
SELECT
    'markets' AS entity,
    (SELECT COUNT(*) FROM silver_markets_candidate) AS candidate_count,
    (SELECT COUNT(*) FROM silver_markets_trusted) AS trusted_count,
    (SELECT COUNT(*) FROM quarantine_markets) AS quarantine_count,
    (SELECT COUNT(*) FROM silver_markets_candidate)
      - (SELECT COUNT(*) FROM silver_markets_trusted)
      - (SELECT COUNT(*) FROM quarantine_markets) AS variance

UNION ALL

SELECT
    'commodities',
    (SELECT COUNT(*) FROM silver_commodities_candidate),
    (SELECT COUNT(*) FROM silver_commodities_trusted),
    (SELECT COUNT(*) FROM quarantine_commodities),
    (SELECT COUNT(*) FROM silver_commodities_candidate)
      - (SELECT COUNT(*) FROM silver_commodities_trusted)
      - (SELECT COUNT(*) FROM quarantine_commodities)

UNION ALL

SELECT
    'prices',
    (SELECT COUNT(*) FROM silver_prices_candidate),
    (SELECT COUNT(*) FROM silver_arrivals_trusted),
    (SELECT COUNT(*) FROM quarantine_prices),
    (SELECT COUNT(*) FROM silver_prices_candidate)
      - (SELECT COUNT(*) FROM silver_arrivals_trusted)
      - (SELECT COUNT(*) FROM quarantine_prices)

UNION ALL

SELECT
    'arrivals',
    (SELECT COUNT(*) FROM silver_arrivals_candidate),
    (SELECT COUNT(*) FROM silver_arrivals_trusted),
    (SELECT COUNT(*) FROM quarantine_arrivals),
    (SELECT COUNT(*) FROM silver_arrivals_candidate)
      - (SELECT COUNT(*) FROM silver_arrivals_trusted)
      - (SELECT COUNT(*) FROM quarantine_arrivals);

entity,candidate_count,trusted_count,quarantine_count,variance
markets,606,594,12,0
commodities,164,156,8,0
prices,96125,87137,1217,7771
arrivals,88220,87137,1083,0


> If any variance is non-zero, stop and investigate before considering Week 6 complete.

## 12. Trusted / Quarantine intersection

No physical record should appear on both sides.

In [0]:
%sql
SELECT 'prices' AS entity, COUNT(*) AS overlap
FROM silver_prices_trusted t
JOIN quarantine_prices q
  ON t.price_record_id = q.price_record_id

UNION ALL

SELECT 'arrivals', COUNT(*)
FROM silver_arrivals_trusted t
JOIN quarantine_arrivals q
  ON t.arrival_record_id = q.arrival_record_id

UNION ALL

SELECT 'markets', COUNT(*)
FROM silver_markets_trusted t
JOIN quarantine_markets q
  ON t.market_id = q.market_id

UNION ALL

SELECT 'commodities', COUNT(*)
FROM silver_commodities_trusted t
JOIN quarantine_commodities q
  ON t.commodity_id = q.commodity_id
 AND t.variety_id = q.variety_id;

entity,overlap
prices,0
arrivals,0
markets,0
commodities,0


## 13. DQ-REL-001 - Daily conformed Candidate

The approved full grain is:

`market_id + commodity_id + variety_id + report_date`

A price/arrival join must use all four keys. Never join on commodity + date only, or another partial key, because that can create many-to-many multiplication.

If `silver_market_daily_candidate` exists, run the following checks.

In [0]:
%sql
SHOW TABLES LIKE 'silver_market_daily_candidate';

database,tableName,isTemporary


In [0]:
%sql
SHOW TABLES LIKE 'silver_market_daily_candidate';

database,tableName,isTemporary


Expected result for the full-grain uniqueness check: **zero rows**.

If your daily Candidate was produced by joining price and arrival data, also compare the row count before and after the join and document any multiplication. Do not hide relationship problems with an inner join.

## 14. DQ-EVT-001 - Streaming event Candidate

`DQ-EVT-001` applies only when your AgriPulse Week-06 implementation has a streaming event Candidate.

Required checks include:
- invalid JSON/schema;
- missing required event fields;
- duplicate `event_id`;
- lateness beyond the approved policy;
- applicable reference/date/price/arrival/unit checks;
- malformed payload context preserved in quarantine.

Do not invent a stream source if your project does not have one yet.

In [0]:
# Optional streaming configuration check.
# Replace STREAM_EVENT_CANDIDATE with the real approved table only if it exists.

STREAM_EVENT_CANDIDATE = None

if STREAM_EVENT_CANDIDATE is None:
    print("No streaming event Candidate configured. DQ-EVT-001 is not executed in this batch notebook.")
else:
    print("Streaming Candidate configured:", STREAM_EVENT_CANDIDATE)

No streaming event Candidate configured. DQ-EVT-001 is not executed in this batch notebook.


## 15. Multi-rule failure proof

One physical record may fail several rules. The quarantine table must retain all applicable IDs in one `rule_ids` field.

For example, an actual record could look like:

`DQ-REF-001, DQ-PRC-001`

Do not manufacture such a record. Run the query below and use whatever the real data produces.

In [0]:
%sql
SELECT
    price_record_id,
    rule_ids,
    reason
FROM quarantine_prices
WHERE instr(rule_ids, ',') > 0
LIMIT 20;

price_record_id,rule_ids,reason
PR0043882,"DQ-OBS-001, DQ-PRC-001",duplicate price business grain occurs 2 times; minimum_price <= modal_price <= maximum_price violated
PRD0023029,"DQ-OBS-001, DQ-PRC-001",duplicate price business grain occurs 2 times; minimum_price <= modal_price <= maximum_price violated
PRD0038903,"DQ-OBS-001, DQ-PRC-001",duplicate price business grain occurs 2 times; minimum_price <= modal_price <= maximum_price violated
PRD0010310,"DQ-OBS-001, DQ-PRC-001",duplicate price business grain occurs 2 times; minimum_price <= modal_price <= maximum_price violated
PRD0030314,"DQ-OBS-001, DQ-PRC-001",duplicate price business grain occurs 2 times; minimum_price <= modal_price <= maximum_price violated
PRD0060512,"DQ-OBS-001, DQ-PRC-001",duplicate price business grain occurs 2 times; minimum_price <= modal_price <= maximum_price violated
PRD0067975,"DQ-OBS-001, DQ-PRC-001",duplicate price business grain occurs 2 times; minimum_price <= modal_price <= maximum_price violated
PRD0072823,"DQ-OBS-001, DQ-PRC-001",duplicate price business grain occurs 2 times; minimum_price <= modal_price <= maximum_price violated
PRD0059921,"DQ-OBS-001, DQ-PRC-001",duplicate price business grain occurs 2 times; minimum_price <= modal_price <= maximum_price violated
PRD0020532,"DQ-OBS-001, DQ-PRC-001",duplicate price business grain occurs 2 times; minimum_price <= modal_price <= maximum_price violated


## 16. Rule-failure counts vs quarantine rows

These are intentionally different metrics. A single quarantined record can contribute to more than one rule-failure count.

In [0]:
%sql
SELECT
    COUNT(*) AS quarantine_price_rows,
    SUM(size(filter(split(rule_ids, ','), x -> trim(x) <> ''))) AS total_rule_failures
FROM quarantine_prices;

quarantine_price_rows,total_rule_failures
1217,1457


## 17. Controlled rerun / replay guidance

For a real correction-and-replay test:

1. Select one quarantined physical record.
2. Record its `physical_record_id`, `_bronze_record_hash`, `run_id`, `rule_ids`, and `reason`.
3. Correct the earliest upstream issue.
4. Create a new input/run lineage.
5. Return the corrected record to Candidate.
6. Re-run the complete applicable rule suite.
7. Preserve the old quarantine evidence.
8. Only the new validated record may enter Trusted Silver.

**Never edit Trusted Silver directly and never promote a quarantined record after checking only the original failed rule.**

In [0]:
%sql
SELECT
    price_record_id,
    rule_ids,
    reason,
    source_file,
    source_file_version,
    source_row_number,
    source_system,
    quarantine_ts,
    rework_status
FROM quarantine_prices
ORDER BY quarantine_ts DESC
LIMIT 1;

price_record_id,rule_ids,reason,source_file,source_file_version,source_row_number,source_system,quarantine_ts,rework_status
PR0000230,DQ-PRC-001,price value is negative,prices.csv,v1.0,230,AGRIPULSE_SYNTHETIC_MODEL_B,2026-09-05T08:58:43.193Z,OPEN


## 18. Final output tables

Expected batch outputs:
- `silver_markets_trusted`
- `silver_commodities_trusted`
- `silver_prices_trusted`
- `silver_arrivals_trusted`
- `quarantine_markets`
- `quarantine_commodities`
- `quarantine_prices`
- `quarantine_arrivals`

The Week-05 Candidate tables remain the source of truth for reconciliation.

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,agripulse_week03_bronze_demo_main,false
default,agripulse_week03_lineage_demo_view,false
default,bronze_arrivals,false
default,bronze_commodities,false
default,bronze_markets,false
default,bronze_prices,false
default,quarantine_arrivals,false
default,quarantine_commodities,false
default,quarantine_markets,false
default,quarantine_prices,false


## 19. Business impact summary

Document the actual issues found in this run.

Suggested interpretation:
- invalid price ordering/bounds can distort price analytics;
- invalid arrival quantities can distort supply/availability analytics;
- broken market/commodity/variety references can produce incomplete or incorrect reporting;
- duplicate business grain can double-count observations;
- invalid dates can place observations in the wrong reporting period;
- unit inconsistencies can make numeric comparisons and conversions unreliable.

Do not claim a business impact that is not relevant to an actual failed rule.

## 20. AI Transparency Note

AI assistance was used to help structure the Week-06 notebook, SQL DQ checks, reusable logic and documentation.

The team must independently:
- execute the notebook in Databricks;
- verify the actual Candidate counts;
- review all DQ results;
- confirm approved configuration values;
- inspect sample failures;
- verify reconciliation and overlap checks;
- accept, modify or reject AI-generated logic.

Do not claim validation until the team has actually executed and checked the results.

## 21. Week-06 exit checklist

- [ ] Four batch Candidate tables exist.
- [ ] `DQ-OBS-001` implemented.
- [ ] `DQ-REF-001` implemented.
- [ ] `DQ-DAT-001` implemented.
- [ ] `DQ-PRC-001` implemented for prices.
- [ ] `DQ-ARR-001` implemented for arrivals.
- [ ] `DQ-UNT-001` implemented.
- [ ] `DQ-REL-001` full-grain safety checked for the daily Candidate.
- [ ] `DQ-EVT-001` implemented if a streaming Candidate exists.
- [ ] Approved configured bounds/unit lists were used where applicable.
- [ ] No private thresholds were guessed.
- [ ] Every applicable failure ID is retained.
- [ ] Trusted and Quarantine tables were created.
- [ ] Candidate = Trusted + Quarantine variance is zero for every batch entity.
- [ ] Trusted/Quarantine intersection is zero.
- [ ] Failed-record evidence captured.
- [ ] DQ results screenshot captured.
- [ ] `data_quality_summary.md` updated.
- [ ] `week06_log.md` updated.
- [ ] AI Transparency Note completed.
- [ ] GitHub commit and push completed.

### Completion standard

Gold must read only from governed Trusted Silver after the Week-06 controls pass.

In [0]:
%sql
SELECT *
FROM silver_markets_trusted
LIMIT 20;

market_record_id,market_id,market_name,raw_market_name,market_type,district_id,district_name,state_code,state_name,latitude,longitude,active_flag,valid_from,source_system,source_record_version,source_row_number,source_file,ingestion_time,obs_id_check,obs_duplicate_check,ref_check,dq_status,rule_ids,reason
MR00001,MKT0001,ADILABAD MANDI 01,ADILABAD MANDI 01,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.07,72.08,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,1,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00002,MKT0002,ADILABAD MANDI 02,ADILABAD MANDI 02,APMC MAIN YARD,TS-D01,ADILABAD,TS,TELANGANA,15.14,72.16,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,2,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00003,MKT0003,ADILABAD MANDI 03,ADILABAD MANDI 03,REGULATED MARKET,TS-D01,ADILABAD,TS,TELANGANA,15.21,72.24,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,3,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00004,MKT0004,ADILABAD MANDI 04,ADILABAD MANDI 04,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.28,72.32,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,4,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00005,MKT0005,ADILABAD MANDI 05,ADILABAD MKT. 05,APMC MAIN YARD,TS-D01,ADILABAD,TS,TELANGANA,15.35,72.4,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,5,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00006,MKT0006,ADILABAD MANDI 06,ADILABAD MANDI 06,REGULATED MARKET,TS-D01,ADILABAD,TS,TELANGANA,15.42,72.48,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,6,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00007,MKT0007,ADILABAD MANDI 07,ADILABAD MANDI 07,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.49,72.56,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,7,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00008,MKT0008,ADILABAD MANDI 08,ADILABAD MANDI 08,APMC MAIN YARD,TS-D01,ADILABAD,TS,TELANGANA,15.56,72.64,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,8,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00009,MKT0009,ADILABAD MANDI 09,ADILABAD MANDI 09,REGULATED MARKET,TS-D01,ADILABAD,TS,TELANGANA,15.63,72.72,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,9,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00010,MKT0010,ADILABAD MANDI 10,ADILABAD MKT. 10,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.7,72.8,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,10,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,


In [0]:
%sql
SELECT *
FROM silver_commodities_trusted
LIMIT 20;

commodity_variety_record_id,commodity_variety_id,commodity_id,commodity_name,raw_commodity_name,category,variety_id,variety_name,raw_variety_name,price_unit,arrival_unit,arrival_to_tonne_factor,base_modal_price_inr_per_quintal,active_flag,source_system,source_row_number,source_file,ingestion_time,obs_id_check,obs_duplicate_check,unit_check,conversion_check,dq_status,rule_ids,reason
CVR00001,CV0001,CMD001,TOMATO,TOMATO,VEGETABLES,VAR0001,STANDARD,STANDARD,INR_PER_QUINTAL,TONNE,1.0,2156.0,true,AGRIPULSE_SYNTHETIC_MODEL_B,1,commodities.csv,2026-07-31T10:07:41.750Z,PASS,PASS,PASS,PASS,PASS,,
CVR00002,CV0002,CMD001,TOMATO,TOMATO,VEGETABLES,VAR0002,GRADE A,GRADE A,INR_PER_QUINTAL,QUINTAL,0.1,2332.0,true,AGRIPULSE_SYNTHETIC_MODEL_B,2,commodities.csv,2026-07-31T10:07:41.750Z,PASS,PASS,PASS,PASS,PASS,,
CVR00003,CV0003,CMD001,TOMATO,TOMATO,VEGETABLES,VAR0003,LOCAL,LOCAL,INR_PER_QUINTAL,TONNE,1.0,2508.0000000000005,true,AGRIPULSE_SYNTHETIC_MODEL_B,3,commodities.csv,2026-07-31T10:07:41.750Z,PASS,PASS,PASS,PASS,PASS,,
CVR00004,CV0004,CMD001,TOMATO,TOMATO,VEGETABLES,VAR0004,PREMIUM,PREMIUM,INR_PER_QUINTAL,TONNE,1.0,2684.0,true,AGRIPULSE_SYNTHETIC_MODEL_B,4,commodities.csv,2026-07-31T10:07:41.750Z,PASS,PASS,PASS,PASS,PASS,,
CVR00005,CV0005,CMD002,ONION,ONION,VEGETABLES,VAR0005,STANDARD,STANDARD,INR_PER_QUINTAL,QUINTAL,0.1,1764.0,true,AGRIPULSE_SYNTHETIC_MODEL_B,5,commodities.csv,2026-07-31T10:07:41.750Z,PASS,PASS,PASS,PASS,PASS,,
CVR00006,CV0006,CMD002,ONION,ONION,VEGETABLES,VAR0006,GRADE A,GRADE A,INR_PER_QUINTAL,TONNE,1.0,1908.0,true,AGRIPULSE_SYNTHETIC_MODEL_B,6,commodities.csv,2026-07-31T10:07:41.750Z,PASS,PASS,PASS,PASS,PASS,,
CVR00007,CV0007,CMD002,ONION,ONION,VEGETABLES,VAR0007,LOCAL,LOCAL,INR_PER_QUINTAL,TONNE,1.0,2052.0,true,AGRIPULSE_SYNTHETIC_MODEL_B,7,commodities.csv,2026-07-31T10:07:41.750Z,PASS,PASS,PASS,PASS,PASS,,
CVR00009,CV0009,CMD003,POTATO,POTATO,VEGETABLES,VAR0009,STANDARD,STANDARD,INR_PER_QUINTAL,TONNE,1.0,1568.0,true,AGRIPULSE_SYNTHETIC_MODEL_B,9,commodities.csv,2026-07-31T10:07:41.750Z,PASS,PASS,PASS,PASS,PASS,,
CVR00010,CV0010,CMD003,POTATO,POTATO,VEGETABLES,VAR0010,GRADE A,GRADE A,INR_PER_QUINTAL,TONNE,1.0,1696.0,true,AGRIPULSE_SYNTHETIC_MODEL_B,10,commodities.csv,2026-07-31T10:07:41.750Z,PASS,PASS,PASS,PASS,PASS,,
CVR00011,CV0011,CMD003,POTATO,POTATO,VEGETABLES,VAR0011,LOCAL,LOCAL,INR_PER_QUINTAL,QUINTAL,0.1,1824.0000000000002,true,AGRIPULSE_SYNTHETIC_MODEL_B,11,commodities.csv,2026-07-31T10:07:41.750Z,PASS,PASS,PASS,PASS,PASS,,


In [0]:
%sql
SELECT *
FROM silver_prices_trusted
LIMIT 20;

price_record_id,market_id,commodity_id,variety_id,report_date,minimum_price,modal_price,maximum_price,price_unit,source_system,source_file_version,source_row_number,source_record_updated_at,load_batch_id,raw_market_name,raw_commodity_name,raw_variety_name,source_file,ingestion_time,obs_id_check,obs_duplicate_check,obs_grain_check,ref_check,date_check,price_order_check,unit_check,dq_status,rule_ids,reason
PR0000001,MKT0354,CMD012,VAR0046,2025-04-18,2894.31,3113.43,3307.82,INR_PER_QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,1,2026-07-01T02:30:00.000Z,BATCH_20260718_01,UJJAIN MANDI 04,JOWAR,GRADE A,prices.csv,2026-07-31T10:12:31.940Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
PR0000002,MKT0268,CMD037,VAR0148,2025-05-14,9630.67,10151.93,10821.8,INR_PER_QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,2,2026-07-02T03:30:00.000Z,BATCH_20260718_01,ERODE MANDI 08,APPLE,PREMIUM,prices.csv,2026-07-31T10:12:31.940Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
PR0000003,MKT0265,CMD012,VAR0048,2025-01-15,4006.73,4045.46,4100.55,INR_PER_QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,3,2026-07-03T04:30:00.000Z,BATCH_20260718_01,ERODE MKT. 05,JOWAR,PREMIUM,prices.csv,2026-07-31T10:12:31.940Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
PR0000004,MKT0061,CMD001,VAR0004,2025-03-03,3054.92,3217.33,3385.57,INR_PER_QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,4,2026-07-04T05:30:00.000Z,BATCH_20260718_01,ANANTAPUR MANDI 01,TOMATO,PREMIUM,prices.csv,2026-07-31T10:12:31.940Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
PR0000005,MKT0406,CMD014,VAR0055,2025-03-19,3123.25,3260.95,3407.57,INR_PER_QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,5,2026-07-05T06:30:00.000Z,BATCH_20260718_01,KOTA MANDI 06,RAGI,LOCAL,prices.csv,2026-07-31T10:12:31.940Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
PR0000006,MKT0034,CMD029,VAR0114,2025-03-04,8913.76,9427.49,9920.36,INR_PER_QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,6,2026-07-06T07:30:00.000Z,BATCH_20260718_01,MAHABUBNAGAR MANDI 04,CORIANDER SEED,GRADE A,prices.csv,2026-07-31T10:12:31.940Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
PR0000007,MKT0383,CMD028,VAR0109,2025-05-21,11676.7,12531.04,13211.78,INR_PER_QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,7,2026-07-07T08:30:00.000Z,BATCH_20260718_01,JAIPUR MANDI 03,TURMERIC,STANDARD,prices.csv,2026-07-31T10:12:31.940Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
PR0000008,MKT0394,CMD011,VAR0043,2025-02-03,2613.04,2769.67,2906.68,INR_PER_QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,8,2026-07-08T09:30:00.000Z,BATCH_20260718_01,JODHPUR MANDI 04,MAIZE,LOCAL,prices.csv,2026-07-31T10:12:31.940Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
PR0000009,MKT0071,CMD036,VAR0141,2025-01-29,2165.97,2260.0,2351.61,INR_PER_QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,9,2026-07-09T10:30:00.000Z,BATCH_20260718_01,CHITTOOR MANDI 01,PAPAYA,STANDARD,prices.csv,2026-07-31T10:12:31.940Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
PR0000010,MKT0107,CMD023,VAR0090,2025-01-06,6744.06,6974.92,7162.39,INR_PER_QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,10,2026-07-10T11:30:00.000Z,BATCH_20260718_01,NELLORE MANDI 07,SUNFLOWER SEED,GRADE A,prices.csv,2026-07-31T10:12:31.940Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,


In [0]:
%sql
SELECT *
FROM silver_arrivals_trusted
LIMIT 20;

arrival_record_id,market_id,commodity_id,variety_id,report_date,arrival_quantity,arrival_unit,source_system,source_file_version,source_row_number,source_record_updated_at,load_batch_id,raw_market_name,raw_commodity_name,raw_variety_name,source_file,ingestion_time,obs_id_check,obs_duplicate_check,obs_grain_check,ref_check,date_check,arrival_check,unit_check,dq_status,rule_ids,reason
AR0000001,MKT0489,CMD003,VAR0012,2025-04-15,184.72,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,1,2026-07-01T04:00:00.000Z,BATCH_20260718_01,AHMEDABAD MANDI 09,POTATO,PREMIUM,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000002,MKT0139,CMD020,VAR0080,2025-02-11,472.294,QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,2,2026-07-02T05:00:00.000Z,BATCH_20260718_01,BALLARI MANDI 09,GROUNDNUT,PREMIUM,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000003,MKT0325,CMD040,VAR0157,2025-06-29,403.579,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,3,2026-07-03T06:00:00.000Z,BATCH_20260718_01,INDORE MKT. 05,GRAPES,STANDARD,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000004,MKT0005,CMD011,VAR0043,2025-04-21,60.794,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,4,2026-07-04T07:00:00.000Z,BATCH_20260718_01,ADILABAD MKT. 05,MAIZE,LOCAL,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000006,MKT0353,CMD038,VAR0150,2025-02-26,50.368,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,6,2026-07-06T09:00:00.000Z,BATCH_20260718_01,UJJAIN MANDI 03,GUAVA,GRADE A,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000007,MKT0592,CMD007,VAR0026,2025-03-26,2500.562,QUINTAL,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,7,2026-07-07T10:00:00.000Z,BATCH_20260718_01,SAMBALPUR MANDI 02,OKRA,GRADE A,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000008,MKT0257,CMD039,VAR0154,2025-03-11,18.995,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,8,2026-07-08T11:00:00.000Z,BATCH_20260718_01,DINDIGUL MANDI 07,POMEGRANATE,GRADE A,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000009,MKT0174,CMD018,VAR0069,2025-05-31,27.222,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,9,2026-07-09T12:00:00.000Z,BATCH_20260718_01,TUMAKURU MANDI 04,URAD DAL,STANDARD,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000010,MKT0317,CMD033,VAR0130,2025-04-20,95.596,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,10,2026-07-10T04:00:00.000Z,BATCH_20260718_01,DEWAS MANDI 07,BANANA,GRADE A,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,
AR0000011,MKT0020,CMD038,VAR0151,2025-06-03,86.729,TONNE,AGRIPULSE_SYNTHETIC_MODEL_B,v1.0,11,2026-07-11T05:00:00.000Z,BATCH_20260718_01,KARIMNAGAR MKT. 10,GUAVA,LOCAL,arrivals.csv,2026-07-31T09:29:34.943Z,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,,


In [0]:
display(spark.table("silver_markets_trusted").limit(10))

market_record_id,market_id,market_name,raw_market_name,market_type,district_id,district_name,state_code,state_name,latitude,longitude,active_flag,valid_from,source_system,source_record_version,source_row_number,source_file,ingestion_time,obs_id_check,obs_duplicate_check,ref_check,dq_status,rule_ids,reason
MR00001,MKT0001,ADILABAD MANDI 01,ADILABAD MANDI 01,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.07,72.08,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,1,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00002,MKT0002,ADILABAD MANDI 02,ADILABAD MANDI 02,APMC MAIN YARD,TS-D01,ADILABAD,TS,TELANGANA,15.14,72.16,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,2,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00003,MKT0003,ADILABAD MANDI 03,ADILABAD MANDI 03,REGULATED MARKET,TS-D01,ADILABAD,TS,TELANGANA,15.21,72.24,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,3,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00004,MKT0004,ADILABAD MANDI 04,ADILABAD MANDI 04,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.28,72.32,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,4,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00005,MKT0005,ADILABAD MANDI 05,ADILABAD MKT. 05,APMC MAIN YARD,TS-D01,ADILABAD,TS,TELANGANA,15.35,72.4,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,5,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00006,MKT0006,ADILABAD MANDI 06,ADILABAD MANDI 06,REGULATED MARKET,TS-D01,ADILABAD,TS,TELANGANA,15.42,72.48,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,6,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00007,MKT0007,ADILABAD MANDI 07,ADILABAD MANDI 07,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.49,72.56,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,7,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00008,MKT0008,ADILABAD MANDI 08,ADILABAD MANDI 08,APMC MAIN YARD,TS-D01,ADILABAD,TS,TELANGANA,15.56,72.64,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,8,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00009,MKT0009,ADILABAD MANDI 09,ADILABAD MANDI 09,REGULATED MARKET,TS-D01,ADILABAD,TS,TELANGANA,15.63,72.72,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,9,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
MR00010,MKT0010,ADILABAD MANDI 10,ADILABAD MKT. 10,SUB YARD,TS-D01,ADILABAD,TS,TELANGANA,15.7,72.8,true,2024-01-01,AGRIPULSE_SYNTHETIC_MODEL_B,1,10,markets.csv,2026-07-31T10:10:00.893Z,PASS,PASS,PASS,PASS,,
